### Data Splitting

In [65]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
import joblib
from scipy.sparse import save_npz
import numpy as np

In [50]:
df = pd.read_csv("cleaned_emails.csv")

In [51]:
df.shape

(4991, 3)

In [52]:
df.isna().sum()

text          0
label_num     0
clean_text    0
dtype: int64

In [53]:
df.duplicated().sum()

np.int64(0)

In [54]:
df.head()

,text,label_num,clean_text
0,Subject: enron methanol ; meter # : 988291\r\n...,0,enron methanol meter 988291 follow note gave m...
1,"Subject: hpl nom for january 9 , 2001\r\n( see...",0,hpl nom january 9 2001 see attached file hplno...
2,"Subject: neon retreat\r\nho ho ho , we ' re ar...",0,neon retreat around wonderful time year neon l...
3,"Subject: photoshop , windows , office . cheap ...",1,photoshop window office cheap main trending ab...
4,Subject: re : indian springs\r\nthis deal is t...,0,indian spring deal book teco pvr revenue under...


In [55]:
X = df["clean_text"]
y = df["label_num"]

In [56]:
# 60% train, 40% temporary holdout
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.4, random_state=42, stratify=y
)

In [57]:
# Split the 40% holdout in half: 20% validation (CV), 20% test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)

In [58]:
print("Train:", len(X_train), "| Val:", len(X_val), "| Test:", len(X_test))
print("Spam ratio -> train: %.3f, val: %.3f, test: %.3f"
      % (y_train.mean(), y_val.mean(), y_test.mean()))

Train: 2994 | Val: 998 | Test: 999
Spam ratio -> train: 0.293, val: 0.293, test: 0.292


In [59]:
X_train.shape, X_val.shape, X_test.shape    

((2994,), (998,), (999,))

### Vectorization

In [60]:
# create the vectorizer and fit it to the training data
vectorizer = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    )

In [61]:
X_train = vectorizer.fit_transform(X_train)
X_val = vectorizer.transform(X_val) 
X_test = vectorizer.transform(X_test)

In [62]:
# inspect
print("Train:", X_train.shape, "| Val:", X_val.shape, "| Test:", X_test.shape)
features = vectorizer.get_feature_names_out()
print("Sample features:", features[:20])
print("'dollar' in vocab:", "dollar" in features)

Train: (2994, 5000) | Val: (998, 5000) | Test: (999, 5000)
Sample features: ['00' '00 00' '00 20' '00 25' '00 30' '00 31' '00 32' '00 463' '00 90'
 '00 daren' '00 deal' '00 delivery' '00 nomination' '00 please' '00 price'
 '00 production' '00 save' '00 thru' '000' '000 00']
'dollar' in vocab: True


In [63]:
X_train[:3].toarray()

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]], shape=(3, 5000))

In [67]:
# save vectorizer and matrices for the model and the Streamlit app
joblib.dump(vectorizer, "tfidf_vectorizer.pkl")
save_npz("model_training_data/X_train.npz", X_train)
save_npz("model_training_data/X_val.npz", X_val)
save_npz("model_training_data/X_test.npz", X_test)

In [68]:
np.save("model_training_data/y_train.npy", y_train.to_numpy())
np.save("model_training_data/y_val.npy", y_val.to_numpy())
np.save("model_training_data/y_test.npy", y_test.to_numpy())